In [ ]:
import os, glob, subprocess
if not os.path.exists('/kaggle/working/Basma'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/us3ph/Basma.git', '/kaggle/working/Basma'], check=True)
os.chdir('/kaggle/working/Basma'); print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
import os, glob, json, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
# the 13 data builds (overviews + manifests) and the 5 native-pixel re-fetch outputs (v6: untouched original-pixel windows)
mans = sorted(m for m in glob.glob('/kaggle/input/**/manifest_*.jsonl', recursive=True) if '/basma-data-' in m)
dirs = sorted({os.path.dirname(m) for m in mans}); print('DATA DIRS', dirs)
assert len(dirs) == 13, f'expected cf/recent/reals/eval + v3ai/v3real + v4a/v4b + v5a-e, found {mans}'
nats = sorted(glob.glob('/kaggle/input/**/native_*.jsonl', recursive=True)); ndirs = sorted({os.path.dirname(n) for n in nats}); print('NATIVE DIRS', ndirs)
assert len(ndirs) == 5, f'expected the 5 native re-fetch outputs, found {nats}'
print(subprocess.run(['df', '-h', '/tmp', '/kaggle/working'], capture_output=True, text=True).stdout)
# leak guard: the v5 result (same images, 422 near-copies of bench / NTIRE images) is in the repo; docs/LEAKGUARD.md too
os.environ['BASMA_EXTRACT_GB'] = '60'; os.environ['BASMA_EXCLUDE'] = 'bench/exclude_ids_v5.txt'; print('BASMA_EXCLUDE', sum(1 for _ in open('bench/exclude_ids_v5.txt')), 'ids')
# baseline scores from v5 (same bench rows and per-image attack seeds)
cache = sorted(glob.glob('/kaggle/input/**/runs/kaggle_v5/baselines/thermostatic.jsonl', recursive=True)) or sorted(glob.glob('/kaggle/input/**/runs/kaggle_v6/baselines/thermostatic.jsonl', recursive=True))
os.environ['BASELINE_CACHE'] = os.path.dirname(cache[0]) if cache else ''; print('BASELINE_CACHE', os.environ['BASELINE_CACHE'] or 'none')
# chained runs: continue from a mounted earlier v6 run (its last/ holds weights + optimizer + step)
prev = sorted(glob.glob('/kaggle/input/**/runs/kaggle_v6/last/opt.pt', recursive=True))
cfg = json.load(open('configs/kaggle_2xT4_v6.json')); cfg['data_dirs'] = dirs + ndirs
cfg['max_hours'] = 5.0   # run 3 must fit in the week's remaining GPU hours: 5 h training + ~1 h scoring
if prev: cfg['resume'] = os.path.dirname(os.path.dirname(prev[-1])); print('RESUME from', cfg['resume'])
json.dump(cfg, open('configs/kaggle_run.json', 'w'), indent=1); print(json.dumps(cfg, indent=1))

In [ ]:
!bash scripts/kaggle_train.sh configs/kaggle_run.json


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('docs/RESULTS.md').read()))
if os.path.exists('runs/kaggle_v6/ntire/NTIRE.md'): display(Markdown(open('runs/kaggle_v6/ntire/NTIRE.md').read()))
